In [3]:
import os
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

MODEL_PATH = "/kaggle/input/datasets/saffigaming/comment-expplanantiom01/codet5_commenst_expla/checkpoint_best"

# 🔥 NEW: Strong reasoning prompt
def build_prompt(code):
    return f"""
You are an expert C++ code reviewer.

Analyze the following code strictly based on LOGIC, not function or variable names.

INSTRUCTIONS:
1. First, explain what each condition or expression actually checks.
2. Then describe what the function really does.
3. If the logic contradicts the function name, report it as an issue.
4. Add clear inline comments to the code.
5. Be precise and avoid generic explanations.

OUTPUT FORMAT:

### COMMENTED CODE
<code with inline comments>

### LOGIC ANALYSIS
<step-by-step explanation of conditions and operations>

### ISSUES
<list any bugs, mismatches, or problems. If none, write "None">

### EXPLANATION
<final clean summary>

CODE:
{code}
"""

def clean_duplicate_code(output):
    """Remove duplicate function blocks if model repeats"""
    parts = output.split("### COMMENTED CODE")
    if len(parts) > 2:
        return "### COMMENTED CODE" + parts[-1]
    return output

def run_test(code_snippet):
    if not os.path.exists(MODEL_PATH):
        print(f"❌ Error: Model path not found at {MODEL_PATH}")
        return

    print("⏳ Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device)
    model.eval()

    # 🔥 Use improved prompt
    prompt = build_prompt(code_snippet)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=512,
        truncation=True,
    ).to(device)

    print("🚀 Generating analysis...")

    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_length=900,
            num_beams=5,
            temperature=0.7,
            no_repeat_ngram_size=3,
            early_stopping=True,
        )

    full_output = tokenizer.decode(output_ids[0], skip_special_tokens=True)

    # 🔥 Clean duplicates
    full_output = clean_duplicate_code(full_output)

    print("\n" + "="*60)
    print("INPUT CODE:")
    print(code_snippet.strip())
    print("="*60)
    print("MODEL OUTPUT:")
    print(full_output.strip())
    print("="*60)

# 2. PASTE YOUR C++ CODE HERE
test_cases = {

# ---------------- EASY ----------------

"EASY 1 (Print Sum Instead of Return)": """
void sumArray(vector<int>& arr) {
    int sum = 0;
    for (int i = 0; i < arr.size(); i++) {
        sum += arr[i];
    }
    cout << sum;
}
""",

"EASY 2 (Odd Check)": """
bool isOdd(int n) {
    return n % 2 == 0;
}
""",

"EASY 3 (Max of Two using Ternary)": """
int maxOfTwo(int a, int b) {
    return (a > b) ? a : b;
}
""",

"EASY 4 (Average of Array)": """
double average(vector<int>& arr) {
    int sum = 0;
    for (int x : arr) {
        sum += x;
    }
    return (double)sum / arr.size();
}
""",

# ---------------- MEDIUM ----------------

"MEDIUM 1 (Bubble Sort)": """
void bubbleSort(int arr[], int n) {
    for (int i = 0; i < n-1; i++) {
        for (int j = 0; j < n-i-1; j++) {
            if (arr[j] > arr[j+1]) {
                int temp = arr[j];
                arr[j] = arr[j+1];
                arr[j+1] = temp;
            }
        }
    }
}
""",

"MEDIUM 2 (Reverse String)": """
string reverseStr(string str) {
    int n = str.length();
    for (int i = 0; i < n / 2; i++)
        swap(str[i], str[n - i - 1]);
    return str;
}
""",

"MEDIUM 3 (Find Maximum Using Loop)": """
int findMax(vector<int>& arr) {
    int mx = arr[0];
    for (int i = 1; i < arr.size(); i++) {
        if (arr[i] > mx) {
            mx = arr[i];
        }
    }
    return mx;
}
""",

"MEDIUM 4 (Linear Search)": """
int findElement(vector<int>& arr, int target) {
    for (int i = 0; i < arr.size(); i++) {
        if (arr[i] == target)
            return i;
    }
    return -1;
}
""",

# ---------------- HARD ----------------

"HARD 1 (Linked List Insert End)": """
void insertAtEnd(Node*& head, int data) {
    Node* newNode = new Node(data);
    if (head == nullptr) {
        head = newNode;
        return;
    }
    Node* temp = head;
    while (temp->next != nullptr) {
        temp = temp->next;
    }
    temp->next = newNode;
}
""",

"HARD 2 (Fibonacci Memoization)": """
int fib(int n, int memo[]) {
    if (memo[n] != -1) return memo[n];
    if (n <= 1) return n;
    memo[n] = fib(n-1, memo) + fib(n-2, memo);
    return memo[n];
}
""",

"HARD 3 (Duplicate Detection using HashSet)": """
bool containsDuplicate(vector<int>& nums) {
    unordered_set<int> seen;
    for (int x : nums) {
        if (seen.count(x)) {
            return true;
        }
        seen.insert(x);
    }
    return false;
}
""",

"HARD 4 (Binary Search)": """
int binarySearch(vector<int>& arr, int target) {
    int left = 0;
    int right = arr.size() - 1;

    while (left <= right) {
        int mid = (left + right) / 2;

        if (arr[mid] == target)
            return mid;

        if (arr[mid] < target)
            left = mid + 1;
        else
            right = mid - 1;
    }

    return -1;
}
""",

"HARD 5 (Factorial Iterative)": """
int factorial(int n) {
    int result = 1;
    for (int i = 2; i <= n; i++) {
        result *= i;
    }
    return result;
}
""",
    
"HARD 7 (Factorial Iterative)": """
int coinChange(vector<int>& coins, int amount) {
     vector<int> dp(amount + 1, amount + 1);
     dp[0] = 0;
     for (int i = 1; i <= amount; i++) {
         for (int coin : coins) {
             if (i - coin >= 0) {
                 dp[i] = min(dp[i], dp[i - coin] + 1);
                }
         }
     }
     return dp[amount] > amount ? -1 : dp[amount];
 }
""",

"HARD 6 (STL Accumulate Sum)": """
int sum(vector<int>& v) {
    return accumulate(v.begin(), v.end(), 0);
}
"""
    


}

# 3. RUN INFERENCE ENGINE
if not os.path.exists(MODEL_PATH):
    print(f"❌ Model not found at {MODEL_PATH}")
else:
    print("⏳ Loading model...")
    tokenizer = AutoTokenizer.from_pretrained(MODEL_PATH)
    model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_PATH)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.to(device).eval()
    for name, code in test_cases.items():
        print(f"\n🚀 Testing {name}...")
        inputs = tokenizer(code.strip(), return_tensors="pt", truncation=True).to(device)
        
        with torch.no_grad():
            out_ids = model.generate(**inputs, max_length=768, num_beams=4)
        
        full_output = tokenizer.decode(out_ids[0], skip_special_tokens=True)
        
        if "===EXPLANATION===" in full_output:
            out_code, out_expl = full_output.split("===EXPLANATION===", 1)
        else:
            out_code, out_expl = full_output, "None"
        print("="*60)
        print(f"RESULT FOR {name}:")
        print("-" * 30)
        print(out_code.strip())
        print(f"\nEXPLANATION: {out_expl.strip()}")
        print("="*60)


⏳ Loading model...


Loading weights:   0%|          | 0/257 [00:00<?, ?it/s]


🚀 Testing EASY 1 (Print Sum Instead of Return)...
RESULT FOR EASY 1 (Print Sum Instead of Return):
------------------------------
void sumArray(vector<int>& arr) {
    int sum = 0; // Initialize sum to 0
    for (int i = 0; i < arr.size(); i++) { // Loop through each element in the vector
        sum += arr[i]; // Add the current element to sum
    }
    cout << sum; // Output the final sum
}

EXPLANATION: The function calculates the sum of all elements in the vector. It iterates through each element, adding it to the sum, and prints the result.

🚀 Testing EASY 2 (Odd Check)...
RESULT FOR EASY 2 (Odd Check):
------------------------------
bool isOdd(int n) {
    // Check if n is odd by testing divisibility by 2
    return n % 2 == 0;
}

EXPLANATION: The function checks if a number is odd by testing divisibility by 2. It returns true if n is odd, otherwise false.

🚀 Testing EASY 3 (Max of Two using Ternary)...
RESULT FOR EASY 3 (Max of Two using Ternary):
------------------------------